In [17]:
import numpy as np

def viterbi_nlp(words, tags, start_p, trans_p, emit_p):
    T = len(words)
    N = len(tags)
    
    # Initialize matrices (using -inf for log-space)
    viterbi_matrix = np.full((N, T), -float('inf'))
    backpointer = np.zeros((N, T), dtype=int)

    # 1. INITIALIZATION:
    # V[s,0]=log P(s)+log P(w_0∣s)
    for s in range(N):
        tag = tags[s]
        if words[0] in emit_p[tag]:
            viterbi_matrix[s, 0] = np.log(start_p[tag]) + np.log(emit_p[tag][words[0]])

    # 2. RECURSION:
    # V[j,t]= max_{i=1}^N (V[i,t−1]+log P(t_j​ ∣t_i​ )+log P(w_t​ ∣ t_j​ ))
    for t in range(1, T):
        for j in range(N): # Current tag
            curr_tag = tags[j]
            for i in range(N): # Previous tag
                prev_tag = tags[i]
                
                # Calculate path probability
                if words[t] in emit_p[curr_tag]:
                    score = viterbi_matrix[i, t-1] + \
                            np.log(trans_p[prev_tag][curr_tag]) + \
                            np.log(emit_p[curr_tag][words[t]])
                    
                    if score > viterbi_matrix[j, t]:
                        viterbi_matrix[j, t] = score
                        backpointer[j, t] = i

    # 3. TERMINATION: Find best ending
    best_last_tag_idx = np.argmax(viterbi_matrix[:, T-1])
    
    # 4. BACKTRACKING: Trace the path
    best_path_indices = [best_last_tag_idx]
    for t in range(T-1, 0, -1):
        best_last_tag_idx = backpointer[best_last_tag_idx, t]
        best_path_indices.append(best_last_tag_idx)
        
    return [tags[i] for i in reversed(best_path_indices)]

In [18]:
# Example Data
my_tags = ['DET', 'NOUN', 'VERB']
my_words = ["the", "bat", "flies"]

# Initial probabilities (π)
start_probs = {'DET': 0.8, 'NOUN': 0.1, 'VERB': 0.1}

# P(next_tag | current_tag)
trans_probs = {
    'DET':  {'DET': 0.01, 'NOUN': 0.9,  'VERB': 0.09},
    'NOUN': {'DET': 0.01, 'NOUN': 0.3,  'VERB': 0.69},
    'VERB': {'DET': 0.4,  'NOUN': 0.5,  'VERB': 0.1}
}

# P(word | tag) 
emit_probs = {
    'DET':  {'the': 1.0},
    'NOUN': {'bat': 0.5, 'flies': 0.5},
    'VERB': {'bat': 0.1, 'flies': 0.9}
}

print(viterbi_nlp(my_words, my_tags, start_probs, trans_probs, emit_probs))

['DET', 'NOUN', 'VERB']


In [13]:
words = ["Janet", "will", "back", "the", "bill"]

tags = ["NNP", "MD", "VB", "JJ", "NN", "RB", "DT"]

start_p = {
    "NNP": 0.2767, "MD": 0.0006, "VB": 0.0031, "JJ": 0.0453, 
    "NN": 0.0449, "RB": 0.0510, "DT": 0.2026
}

trans_p = {
    "NNP": {"NNP": 0.3777, "MD": 0.0110, "VB": 0.0009, "JJ": 0.0084, "NN": 0.0584, "RB": 0.0090, "DT": 0.0025},
    "MD":  {"NNP": 0.0008, "MD": 0.0002, "VB": 0.7968, "JJ": 0.0005, "NN": 0.0008, "RB": 0.1698, "DT": 0.0041},
    "VB":  {"NNP": 0.0322, "MD": 0.0005, "VB": 0.0050, "JJ": 0.0837, "NN": 0.0615, "RB": 0.0514, "DT": 0.2231},
    "JJ":  {"NNP": 0.0366, "MD": 0.0004, "VB": 0.0001, "JJ": 0.0733, "NN": 0.4509, "RB": 0.0036, "DT": 0.0036},
    "NN":  {"NNP": 0.0096, "MD": 0.0176, "VB": 0.0014, "JJ": 0.0086, "NN": 0.1216, "RB": 0.0177, "DT": 0.0068},
    "RB":  {"NNP": 0.0068, "MD": 0.0102, "VB": 0.1011, "JJ": 0.1012, "NN": 0.0120, "RB": 0.0728, "DT": 0.0479},
    "DT":  {"NNP": 0.1147, "MD": 0.0021, "VB": 0.0002, "JJ": 0.2157, "NN": 0.4744, "RB": 0.0102, "DT": 0.0017}
}

emit_p = {
    "NNP": {"Janet": 0.000032},
    "MD":  {"will": 0.308431},
    "VB":  {"will": 0.000028, "back": 0.000672, "bill": 0.000028},
    "JJ":  {"back": 0.000340},
    "NN":  {"will": 0.000200, "back": 0.000223, "bill": 0.002337},
    "RB":  {"back": 0.010446},
    "DT":  {"the": 0.506099}
}

print(viterbi_nlp(words, tags, start_p, trans_p, emit_p))

['NNP', 'MD', 'VB', 'DT', 'NN']


In [24]:
import nltk
from nltk.corpus import treebank
from nltk.tag import HiddenMarkovModelTrainer
nltk.download('treebank')

# Get training data
train_data = treebank.tagged_sents()[:3000]

# The trainer handles the counting and matrix building for you
trainer = HiddenMarkovModelTrainer()
tagger = trainer.train_supervised(train_data)

# The .tag() method executes the Viterbi algorithm


[nltk_data] Downloading package treebank to /home/shk/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


In [ ]:
import numpy as np
from collections import defaultdict

class ViterbiTagger:
    def __init__(self):
        self.tags = []
        self.start_p = {}
        self.trans_p = defaultdict(lambda: defaultdict(float))
        self.emit_p = defaultdict(lambda: defaultdict(float))

    def train(self, corpus):
        """
        corpus: List of list of tuples [(word, tag), ...]
        """
        tag_counts = defaultdict(int)
        transition_counts = defaultdict(lambda: defaultdict(int))
        emission_counts = defaultdict(lambda: defaultdict(int))
        start_counts = defaultdict(int)

        for sentence in corpus:
            for i, (word, tag) in enumerate(sentence):
                tag_counts[tag] += 1
                emission_counts[tag][word] += 1
                
                if i == 0:
                    start_counts[tag] += 1
                else:
                    prev_tag = sentence[i-1][1]
                    transition_counts[prev_tag][tag] += 1

        self.tags = list(tag_counts.keys())
        total_sentences = len(corpus)

        # Convert counts to Log-Probabilities to avoid underflow
        for tag in self.tags:
            # Start Probabilities
            self.start_p[tag] = np.log((start_counts[tag] + 1) / (total_sentences + len(self.tags)))
            
            # Transition Probabilities
            for next_tag in self.tags:
                count = transition_counts[tag][next_tag]
                self.trans_p[tag][next_tag] = np.log((count + 1) / (tag_counts[tag] + len(self.tags)))
            
            # Emission Probabilities
            for word, count in emission_counts[tag].items():
                self.emit_p[tag][word] = np.log((count + 1) / (tag_counts[tag] + len(emission_counts[tag])))

    def tag(self, sentence):
        T = len(sentence)
        N = len(self.tags)
        
        viterbi_matrix = np.full((N, T), -float('inf'))
        backpointer = np.zeros((N, T), dtype=int)

        # Initialization
        for s in range(N):
            tag = self.tags[s]
            # Use a small default log value for unknown words
            emission = self.emit_p[tag].get(sentence[0], -20.0) 
            viterbi_matrix[s, 0] = self.start_p[tag] + emission

        # Recursion
        for t in range(1, T):
            for j in range(N):
                curr_tag = self.tags[j]
                emission = self.emit_p[curr_tag].get(sentence[t], -20.0)
                
                # Vectorized search for max probability from previous states
                scores = viterbi_matrix[:, t-1] + \
                         np.array([self.trans_p[self.tags[i]][curr_tag] for i in range(N)]) + \
                         emission
                
                viterbi_matrix[j, t] = np.max(scores)
                backpointer[j, t] = np.argmax(scores)

        # Backtracking
        best_path_idx = [np.argmax(viterbi_matrix[:, T-1])]
        for t in range(T-1, 0, -1):
            best_path_idx.append(backpointer[best_path_idx[-1], t])
            
        return [self.tags[i] for i in reversed(best_path_idx)]

# --- EXAMPLE USAGE ---


# 2. Initialize and Train
tagger = ViterbiTagger()
tagger.train(train_data)

# 3. Test on a new sentence
test_sentence = ["the", "bat", "flies"]
predicted_tags = tagger.tag(test_sentence)

print(f"Sentence: {test_sentence}")
print(f"Tags:     {predicted_tags}")

Sentence: ['the', 'bat', 'flies']
Tags:     ['DT', 'NN', 'IN']
